We want to see how many days in the month is the sst above 28 Celcius? this is the average for the summer months.

In [27]:
# import library
import ee
import geemap
import geopandas as gpd
import pandas as pd
import time
import datetime


# authenticate & initilialize earth engine
ee.Authenticate()
ee.Initialize(project='ee-sst-j-felix')

In [28]:
# get the boundary for an office
# load the shapefile
office = gpd.read_file('/content/drive/MyDrive/BIENPESCA/shapefiles/office_points_latest.shp')

office.columns = ["state_id",  "state",    "office_id",  "office",   "locality",  "cvegeo",   "status",   "stat_abr",  "mun_id",
                          "local_id",  "climate",  "latitud", "longitud",  "altitud",  "letter_id",  "population",  "male_pop",  "feml_pop",
                          "opccupied_households",  "obs_id",   "municipio",  "geometry"]


In [59]:
# make the shapefile into an ee object
office_ee = geemap.gdf_to_ee(office)

In [60]:
# 2. Create 50 km buffer around each office with a simplified shape
buffered_offices = office_ee.map(
    lambda f: f.buffer(50000).simplify(5000)
)


In [61]:
# GE info
# https://developers.google.com/earth-engine/datasets/catalog/NOAA_CDR_OISST_V2_1#bands
# 27830 meters pixel size
# 0.01 scale
# measurement in Celsius
# get the image collection (filter for date)
sst = ee.ImageCollection('NOAA/CDR/OISST/V2_1').filterDate('2006-01-01', '2024-12-31').select('sst')



In [62]:
# Mark SST images as hot (>28°C) at the regional (buffer) scale
def is_hot_day(img):
    return img.set('date', img.date().format('YYYY-MM-dd'))

sst = sst.map(is_hot_day)

In [68]:
# Loop over time and extract hot-day flags per buffer
years = list(range(2006, 2025))
months = list(range(1, 13))
results = []

for year in years:
    for month in months:
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, 'month')

        sst_month = sst.filterDate(start, end)

        def count_hot_days(feature):
            geom = feature.geometry()
            office_id = feature.get('office_id')

            def daily_flag(img):
                mean_sst = img.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=geom,
                    scale=5000,
                    maxPixels=1e9
                ).get('sst')

                # Only run .gt(28) if mean_sst is not null
                is_hot = ee.Algorithms.If(
                    mean_sst,
                    ee.Number(mean_sst).gt(28),
                    0  # Not hot if SST is missing
                )

                return ee.Feature(None, {
                    'office_id': office_id,
                    'year': year,
                    'month': month,
                    'date': img.date().format('YYYY-MM-dd'),
                    'is_hot': is_hot
                })
            # Corrected indentation
            hot_flags = sst_month.map(daily_flag)
            return hot_flags

        # Apply per buffer zone
        monthly_flags = buffered_offices.map(count_hot_days).flatten()
        results.append(monthly_flags)

In [69]:
# 6. Combine and export the result to Drive
combined_results = ee.FeatureCollection(results).flatten()

In [70]:
# see a sample of 20 observations
combined_results.limit(20).getInfo()


{'type': 'FeatureCollection',
 'columns': {},
 'features': [{'type': 'Feature',
   'geometry': None,
   'id': '0_0_20060101',
   'properties': {'date': '2006-01-01',
    'is_hot': 0,
    'month': 1,
    'office_id': 101,
    'year': 2006}},
  {'type': 'Feature',
   'geometry': None,
   'id': '0_0_20060102',
   'properties': {'date': '2006-01-02',
    'is_hot': 0,
    'month': 1,
    'office_id': 101,
    'year': 2006}},
  {'type': 'Feature',
   'geometry': None,
   'id': '0_0_20060103',
   'properties': {'date': '2006-01-03',
    'is_hot': 0,
    'month': 1,
    'office_id': 101,
    'year': 2006}},
  {'type': 'Feature',
   'geometry': None,
   'id': '0_0_20060104',
   'properties': {'date': '2006-01-04',
    'is_hot': 0,
    'month': 1,
    'office_id': 101,
    'year': 2006}},
  {'type': 'Feature',
   'geometry': None,
   'id': '0_0_20060105',
   'properties': {'date': '2006-01-05',
    'is_hot': 0,
    'month': 1,
    'office_id': 101,
    'year': 2006}},
  {'type': 'Feature',
   'g

In [71]:
task = ee.batch.Export.table.toDrive(
    collection=combined_results,
    description='sst_hot_days_50km_buffers',
    fileFormat='CSV'
)
task.start()

print("🚀 Export started! Check Earth Engine Tasks or Google Drive.")

🚀 Export started! Check Earth Engine Tasks or Google Drive.
